In [6]:
import pandas as pd
import numpy as np
import os
import json

def get_dataset_info_by_species_name(workdir: str, species_name: str) -> pd.DataFrame:
    """根据菌种的名字，返回对应的dataset的信息
    """
    species_name = species_name.replace(' ', '_').lower()
    df_dataset = pd.read_csv(os.path.join(workdir, species_name, 'dataset.tsv'), sep='\t')
    
    return df_dataset


def get_sample_info_by_species_name(workdir: str, species_name: str) -> pd.DataFrame:
    """根据菌种的名字，返回对应的sample的信息
    """
    species_name = species_name.replace(' ', '_').lower()
    df_sample = pd.read_csv(os.path.join(workdir, species_name, 'sample.tsv'), sep='\t')
    
    return df_sample


def get_exp_by_gene_id(workdir: str, species_name: str, gene_id: str) -> pd.DataFrame:
    """根据菌种的名字，返回对应的gene的表达量表
    """
    species_name = species_name.replace(' ', '_').lower()
    df_gene_exp = pd.read_csv(os.path.join(workdir, species_name, 'exp_gsm.tsv'), sep='\t')
    df = df_gene_exp[df_gene_exp['Gene id'] == gene_id]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)
    # df = df.loc[gene_id, :]

    return df


def get_exp_fname_by_gene_id(workdir: str, species_name: str, gene_id: str) -> pd.DataFrame:
    """根据菌种的名字，返回对应的gene的表达量表
    """
    species_name = species_name.replace(' ', '_').lower()
    df_gene_exp = pd.read_csv(os.path.join(workdir, species_name, 'exp_fname.tsv'), sep='\t')
    df = df_gene_exp[df_gene_exp['Gene id'] == gene_id]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)
    
    # 将df这个dataframe转为字典格式
    df = df.to_dict(orient='records')

    return df

**根据Gene ID获取改基因的Go注释，存在一对多的情况**

In [1]:
# def get_gene_info_by_gene_id(workdir: str, species_name: str, gene_id: str) -> dict:
#     """根据菌种的名字，及gene的id，返回对应的gene的go注释
#     """
#     species_name = species_name.replace(' ', '_').lower()    
#     df_gene_info = pd.read_csv(os.path.join(workdir, species_name, 'go.tsv'), sep='\t')
#     df_info = df_gene_info[df_gene_info['Gene'] == gene_id]

#     # 将相同基因的go注释合并
#     df_info = df_info.groupby('Gene').agg(lambda x: list(x)).reset_index()
#     df_info = df_info.to_dict(orient='records')

#     return df_info


def get_gene_go_info_by_gene_id(workdir: str, species_name: str, gene_id: str) -> dict:
    """返回给定物种和基因ID的基因GO注释。
    """
    # 将物种名转换为小写，并替换空格为下划线
    species_name = species_name.lower().replace(' ', '_')
    # 获取注释信息文件路径
    go_info_path = os.path.join(workdir, species_name, 'go.tsv')
    df_go_info = pd.read_csv(go_info_path, sep='\t')
    
    # 筛选出指定基因ID的注释信息，将GO注释合并到一个列表中
    df_info = df_go_info.loc[df_go_info['Gene'] == gene_id].groupby('Gene')['GOID'].apply(list).reset_index()
    # 将结果转换为字典格式并返回
    return df_info.to_dict(orient='records')

In [99]:
# 不是最优秀的代码，但是能够工作
def get_pathway_by_gene_id(workdir,species_name,gene_id):
    """根据Gene ID，返回对应的pathway的名字与ID号
    """
    species_name = species_name.replace(' ', '_').lower()
    pathway_dict = {}

    # 读取kegg.tsv
    pathway_path = os.path.join(workdir, species_name, 'kegg.tsv')
    df_pathway = pd.read_csv(pathway_path, sep='\t')

    # 读取pathway_id.tsv，并设置表头
    pathway_id_path = os.path.join(workdir, species_name, 'pathway_id.tsv')
    df_pathway_id = pd.read_csv(pathway_id_path, sep='\t')
    df_pathway_id.columns = ['Id', 'Pathway'] 

    # 如果gene_id不在df_pathway中，返回N/A
    if not df_pathway['Gene'].isin([gene_id]).any():
        pathway_name = 'N/A'
    else:
        pathway_name = df_pathway.loc[df_pathway['Gene'] == gene_id, 'Pathway'].tolist()

    # 如果pathway_name不在df_pathway_id中，返回N/A
    if not df_pathway_id['Pathway'].isin(pathway_name).any():
        pathway_id = 'N/A'
    else:
        pathway_id = df_pathway_id.loc[df_pathway_id['Pathway'].isin(pathway_name), 'Id'].tolist()

    #  将pathway_name和pathway_id添加到字典中
    pathway_dict['pathway_name'] = pathway_name
    pathway_dict['pathway_id'] = pathway_id

    return pathway_dict

# get_pathway_by_gene_id(workdir,species_name,gene_id)

**根据基因ID,返回包含通路名称、通路ID、NCBI ID和蛋白质名称的字典.**

In [16]:
def get_gene_info_by_gene_id(workdir: str, species_name: str, gene_id: str) -> dict:
    """Returns a dictionary containing pathway name, pathway ID, NCBI ID, and protein name for a given gene ID.

    Args:
        workdir: The root directory containing the data files.
        species_name: The name of the species, used to determine the location and name of the data files.
        gene_id: The ID of the gene, used to lookup related information in the KEGG pathway database and NCBI ID database.

    Returns:
        A dictionary containing pathway name, pathway ID, NCBI ID, and protein name. If no corresponding information is found,
        the respective value will be 'N/A'.
    """
    # Format species_name and gene_id
    species_name = species_name.lower().replace(' ', '_')
    gene_id = gene_id.upper()

    # files path
    kegg_file = os.path.join(workdir, species_name, 'kegg.tsv')
    pathway_file = os.path.join(workdir, species_name, 'pathway_id.tsv')
    ncbi_id_file = os.path.join(workdir, species_name, 'ncbi_id.tsv')
    location_file = os.path.join(workdir, species_name, 'location.tsv')
    go_file = os.path.join(workdir, species_name, 'go.tsv')

    # Read data files
    df_kegg = pd.read_csv(kegg_file, sep='\t')
    df_pathway = pd.read_csv(pathway_file, sep='\t', names=['Id', 'Pathway'])
    df_ncbi_id = pd.read_csv(ncbi_id_file, sep='\t', names=['NCBI ID', 'Gene ID', 'Protein'])
    df_location = pd.read_csv(location_file, sep='\t')
    df_go = pd.read_csv(go_file, sep='\t')

    # Get pathway name and ID as a dictionary
    pathway_name_list = df_kegg.loc[df_kegg['Gene'] == gene_id, 'Pathway'].tolist()
    pathway_name = pathway_name_list if pathway_name_list else 'N/A'
    pathway_id_list = df_pathway.loc[df_pathway['Pathway'].isin(pathway_name_list), 'Id'].tolist()
    pathway_id = pathway_id_list if pathway_id_list else 'N/A'
    pathway_dict = dict(zip(pathway_id, pathway_name))

    # Get NCBI ID and protein name
    ncbi_id = df_ncbi_id.loc[df_ncbi_id['Gene ID'] == 'mtm:' + gene_id, 'NCBI ID'].tolist()
    ncbi_id = ncbi_id[0] if ncbi_id else 'N/A'
    protein = df_ncbi_id.loc[df_ncbi_id['Gene ID'] == 'mtm:' + gene_id, 'Protein'].tolist()
    protein = protein[0] if protein else 'N/A'

    # Get gene location information
    gene_location = df_location.loc[df_location['Geneid'] == gene_id, ['Chr', 'Start', 'End', 'Strand', 'Length']].to_dict(orient='records')

    # Get GO information
    go_id = df_go.loc[df_go['Gene'] == gene_id, ['GOID', 'DEFINITION', 'ONTOLOGY', 'TERM']].to_dict(orient='records')
    go_id = go_id if go_id else 'N/A'


    # Create and return the gene dictionary
    dict_gene = {
        gene_id: {
            'location': gene_location,
            'NCBI ID': ncbi_id,
            'Protein': protein,
            'Pathway': pathway_dict,
            'GO': go_id,
        }
    }

    return dict_gene

get_gene_info_by_gene_id(workdir,species_name,gene_id)

{'MYCTH_2304931': {'location': [{'Chr': 'NC_016474.1',
    'Start': 3262765,
    'End': 3264160,
    'Strand': '-',
    'Length': 1396}],
  'NCBI ID': 'ncbi-geneid:11507874',
  'protein': 'hypothetical protein',
  'Pathway': {'mtm01100': 'Glycolysis / Gluconeogenesis',
   'mtm01110': 'Fructose and mannose metabolism',
   'mtm01200': 'Inositol phosphate metabolism',
   'mtm01230': 'Metabolic pathways',
   'mtm00010': 'Biosynthesis of secondary metabolites',
   'mtm00051': 'Carbon metabolism',
   'mtm00562': 'Biosynthesis of amino acids'},
  'GO': [{'GOID': 'GO:0004807',
    'DEFINITION': 'Catalysis of the reaction: D-glyceraldehyde 3-phosphate = glycerone phosphate.',
    'ONTOLOGY': 'MF',
    'TERM': 'triose-phosphate isomerase activity'},
   {'GOID': 'GO:0003824',
    'DEFINITION': 'Catalysis of a biochemical reaction at physiological temperatures. In biologically catalyzed reactions, the reactants are known as substrates, and the catalysts are naturally occurring macromolecular subst